대부분의 데이터셋을 허깅페이스 허브에 있는 데이터셋을 받아오고 전처리해서 쓰도록 함. `datasets` 라이브러리를 통해서 수행할 수 있는데 그 뿐만 아니라 로컬에 있는 데이터로 같은 형식으로 처리할 수 있다.

주로 사용하게 될 것들

1) 로드하기 : `load_dataset`
2) 조회하기 : `take`, `select`
3) 전처리하기 : `map`
4) 나누기 : `train_test_split`
5) 올리기 : `push`

In [ ]:
import datasets
from datasets import load_dataset

### 1. 허브에서 이용한 로드하기

`load_dataset`는 다양한 소스에서 데이터를 받아서 올릴 수 있는데 우선 HF 허브를 출처로 할 때...

In [2]:
repository_id = "jtatman/python-code-dataset-500k"
dataset_dict = load_dataset(repository_id)

데이터셋 로드를 하면 `/home/$USER/` 아래에 캐시 폴더가 있고 여기에 데이터셋을 다운받게 됨. 정확히는 `cache` 폴더 아래에 `huggingface/datasets` 폴더 안에 다운받게 됨 : `/purestorage/AILAB/AI_1/tyk/0_Software/cache/huggingface/datasets/jtatman___python-code-dataset-500k`

In [3]:
!ls /purestorage/AILAB/AI_1/tyk/0_Software/cache/huggingface/datasets/jtatman___python-code-dataset-500k/default/0.0.0

print("----------------------------")

!ls /purestorage/AILAB/AI_1/tyk/0_Software/cache/huggingface/datasets/jtatman___python-code-dataset-500k/default/0.0.0/060ad3df88a6ba5f5546c622652290f38e73ceba

060ad3df88a6ba5f5546c622652290f38e73ceba
060ad3df88a6ba5f5546c622652290f38e73ceba.incomplete_info.lock
060ad3df88a6ba5f5546c622652290f38e73ceba_builder.lock
----------------------------
dataset_info.json
python-code-dataset-500k-train-00000-of-00002.arrow
python-code-dataset-500k-train-00001-of-00002.arrow


여기에 arrow 파일과 dataset 정보가 담긴 json 파일이 다운받아진 것을 알 수 있다. 이런 arrow 파일은 판다스를 써서 확인해볼 수도 있음.

In [4]:
import pyarrow as pa

# binary mode로 열기
with open("/purestorage/AILAB/AI_1/tyk/0_Software/cache/huggingface/datasets/jtatman___python-code-dataset-500k/default/0.0.0/060ad3df88a6ba5f5546c622652290f38e73ceba/python-code-dataset-500k-train-00000-of-00002.arrow", "rb") as f:
    reader = pa.ipc.RecordBatchStreamReader(f)
    table = reader.read_all()

# Pandas로 변환
df = table.to_pandas()
print(df.head())

                                              output  \
0  Here is an example of a nested loop in Python ...   
1  The given problem can be solved by iterating t...   
2  Here's an example of code that attempts to sol...   
3  Here is an implementation of the function in P...   
4  Here's a possible implementation of the method...   

                                         instruction  \
0  Create a nested loop to print every combinatio...   
1  Write a function to find the number of distinc...   
2  Write code that removes spaces and punctuation...   
3  Write a function that checks if a given number...   
4  Write a method for a string class which replac...   

                                              system  
0  You are a Python code analyst, evaluating scri...  
1  As a Python code composer, you craft elegant a...  
2  As a Python code analyst, you investigate and ...  
3  As a Python code composer, you craft elegant a...  
4  As a Python code translator, you convert algor..

In [5]:
df.columns, len(df)

(Index(['output', 'instruction', 'system'], dtype='object'), 217000)

`load_dataset`이 반환하는 객체는 `DatasetDict`임. `DatasetDict`는 다시 split으로 구분되는 `Dataset`을 담고 있음. `Dataset` 클래스는 `features` key에 각 행에 해당되는 실제 피처를 칼럼에 담고 있음.

In [6]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'system'],
        num_rows: 559515
    })
})

In [7]:
dataset_dict.keys()

dict_keys(['train'])

In [8]:
dataset_dict['train']

Dataset({
    features: ['output', 'instruction', 'system'],
    num_rows: 559515
})

In [9]:
dataset_dict['train']['output'][:3]

['Here is an example of a nested loop in Python to print every combination of numbers between 0-9, excluding any combination that contains the number 5 or repeating digits:\n\n```python\nfor i in range(10):  # First digit\n    for j in range(10):  # Second digit\n        for k in range(10):  # Third digit\n            # Checking for the conditions\n            if i != 5 and j != 5 and k != 5 and i != j and i != k and j != k:\n                print(i, j, k)\n```\n\nThis code will generate and print every combination of three digits between 0-9 that do not contain the number 5 and do not have any repeating digits.',
 "The given problem can be solved by iterating through each cell of the matrix and converting the state of the cell into a string. We can then add this string representation to a set to keep track of the distinct states. Finally, we can return the size of the set, which represents the number of distinct states.\n\nHere's the correct code to solve the problem:\n\n```python\nde

In [10]:
dataset_dict['train'].features

{'output': Value(dtype='string', id=None),
 'instruction': Value(dtype='string', id=None),
 'system': Value(dtype='string', id=None)}

In [11]:
# 전체말고 일부만 로드하기
# split과 인덱싱을 이용해서 일부만 받을 수 있다.
# 앞에서 다운받았다면 캐시에서 바로 로드해줌
repository_id = "jtatman/python-code-dataset-500k"
dataset = load_dataset(repository_id, split='train[:10]')

# Percent slicing
dataset = load_dataset(repository_id, split='train[50%:52%]')

In [12]:
dataset

Dataset({
    features: ['output', 'instruction', 'system'],
    num_rows: 11190
})

In [13]:
print(dataset)

Dataset({
    features: ['output', 'instruction', 'system'],
    num_rows: 11190
})


In [14]:
print(dataset[0])

{'output': 'Compile a SASS source file and output it.', 'instruction': "Make a summary of the following Python 3 code\ndef compile_sass(input_file, output_file):\n    from .modules import sass\n\n    if not isinstance(input_file, str):\n        raise RuntimeError('SASS compiler takes only a single input file.')\n\n    return {\n        'dependencies_fn': sass.sass_dependencies,\n        'compiler_fn': sass.sass_compile,\n        'input': input_file,\n        'output': output_file,\n        'kwargs': {},\n    }", 'system': 'You are a Python code analyst, evaluating scripts for potential improvements.'}


git 명령어로 데이터를 받아서 로컬에 저장

In [1]:
name = "pokemon"

In [2]:
!git clone https://huggingface.co/datasets/huggan/{name} && (cd {name} && git lfs pull)

Cloning into 'pokemon'...
remote: Enumerating objects: 13, done.
remote: Total 13 (delta 0), reused 0 (delta 0), pack-reused 13 (from 1)
Unpacking objects: 100% (13/13), 1.77 KiB | 22.00 KiB/s, done.


In [3]:
!pwd

/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model


In [5]:
import pyarrow.parquet as pq
from io import BytesIO
from pathlib import Path
from tqdm import tqdm
from PIL import Image

i = 0
for table in Path(f"{name}/data").glob("*.parquet"):
    for row in tqdm(pq.read_table(table)[0]):
        Image.open(BytesIO(row["bytes"].as_py())).save(f"{name}/{i:04d}.jpg")
        i += 1

100%|██████████| 7357/7357 [00:41<00:00, 176.36it/s]


hf_hub_download를 써서 직접 다운받아서 로드할 수 있음

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "jtatman/python-code-dataset-500k"
filename = "data/train-00000-of-00002.parquet"
cache_dir = "/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/hugging_dataset/out"

file_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset", cache_dir=cache_dir)

data/train-00000-of-00002.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

In [19]:
!ls /purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/hugging_dataset/out/datasets--jtatman--python-code-dataset-500k/snapshots/060ad3df88a6ba5f5546c622652290f38e73ceba/data

train-00000-of-00002.parquet


In [20]:
import pandas as pd 
df = pd.read_parquet('/purestorage/AILAB/AI_1/tyk/3_CUProjects/datasets--jtatman--python-code-dataset-500k/snapshots/060ad3df88a6ba5f5546c622652290f38e73ceba/data/train-00000-of-00002.parquet', engine='pyarrow')
print(df)

                                                   output  \
0       Here is an example of a nested loop in Python ...   
1       The given problem can be solved by iterating t...   
2       Here's an example of code that attempts to sol...   
3       Here is an implementation of the function in P...   
4       Here's a possible implementation of the method...   
...                                                   ...   
279753  Concatenates the input files into a single out...   
279754  Copies any files matching filespec from src_di...   
279755  Minifies the input javascript files to the out...   
279756  Splits a large CSS file into several smaller f...   
279757  def compile_less(input_file, output_file):\n  ...   

                                              instruction  \
0       Create a nested loop to print every combinatio...   
1       Write a function to find the number of distinc...   
2       Write code that removes spaces and punctuation...   
3       Write a functio

### 2. 조회하기

In [21]:
# 조회
dataset_dict['train'].select([1])

Dataset({
    features: ['output', 'instruction', 'system'],
    num_rows: 1
})

In [22]:
dataset_dict['train'].take(10)  

Dataset({
    features: ['output', 'instruction', 'system'],
    num_rows: 10
})

### 3. 전처리하기

In [ ]:
# shuffle 후 일부 select하기

빠르게 하려면 `batched`나 `num_proc`을 쓰도록 함.

batched 옵션으로 한다면 지정한 함수의 인풋으로 batch 단위 데이터가 들어감.

예를 들어 

batch = {
    "sql_prompt":  ["SELECT … FROM …",  "SELECT …",  "SELECT …"],
    "sql_context": ["CREATE TABLE …",    "CREATE TABLE …", "CREATE TABLE …"],
    "sql":         ["SELECT …",          "SELECT …",       "SELECT …"],
    # …(다른 컬럼도 같은 길이의 리스트)
}

In [ ]:
# batched 옵션을 줄 때는 지정한 함수의 인풋으로 batch 단위 데이터가 들어감.
dataset = dataset.map(create_conversation, remove_columns=dataset.features,batched=True)

### 4. 나누기

### 5. 올리기

허브는 깃허브와 똑같기 때문에 commit-push하는 식으로 올라감.

In [ ]:
# 미리 로그인 해두기
from huggingface_hub import login, create_repo

login(token=args.hf_token)
create_repo(args.output_repo, repo_type="dataset", exist_ok=True)

dataset.push_to_hub(
    output_repo
    )


In [ ]:
# 올리고 나서 불러오기
# 읽을 때 commit id를 지정해서 읽을 수 있음

ds = load_dataset(output_repo, split="train", revision="9b865ef") 

### 6. 로컬 데이터셋 불러서 처리하기


허깅페이스 허브가 아닌 특정 확장자의 파일을 직접 지정해서 받을 수 있음. 확장자마다 지정하는 `path` 형식이 다른거 체크.

`load_dataset("csv", data_files="my_file.csv")`

`load_dataset("text", data_files="my_file.txt")`

`load_dataset("json", data_files="my_file.jsonl")`

`load_dataset("pandas", data_files="my_dataframe.pkl")`

In [24]:
#!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
#!unzip drugsCom_raw.zip

from datasets import load_dataset

# tsv 파일은 콤마 대신 탭으로 구분 -> csv 파일 로드하듯이하고 \t로 구분자를 지정
data_files = {"train": "/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/datasets/drugsComTrain_raw.tsv", "test": "/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/datasets/drugsComTest_raw.tsv"}
# \t is the tab character in Python
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [2]:
drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
# Peek at the first few examples
drug_sample[:3]

{'Unnamed: 0': [87571, 178045, 80482],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic'],
 'condition': ['Gout, Acute', 'ibromyalgia', 'Inflammatory Conditions'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',
  '"I have been taking Mobic for over a year with no side effects other than 

In [3]:
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

In [5]:
len(drug_dataset["train"].unique("Unnamed: 0"))

161297

In [ ]:
# 'Unnamed: 0 => 'patient_id'로 바꾸기
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [7]:
drug_dataset["train"].unique("drugName")

['Valsartan',
 'Guanfacine',
 'Lybrel',
 'Ortho Evra',
 'Buprenorphine / naloxone',
 'Cialis',
 'Levonorgestrel',
 'Aripiprazole',
 'Keppra',
 'Ethinyl estradiol / levonorgestrel',
 'Topiramate',
 'L-methylfolate',
 'Pentasa',
 'Dextromethorphan',
 'Nexplanon',
 'Liraglutide',
 'Trimethoprim',
 'Amitriptyline',
 'Lamotrigine',
 'Nilotinib',
 'Atripla',
 'Trazodone',
 'Etonogestrel',
 'Etanercept',
 'Tioconazole',
 'Azithromycin',
 'Eflornithine',
 'Daytrana',
 'Ativan',
 'Imitrex',
 'Sertraline',
 'Toradol',
 'Viberzi',
 'Mobic',
 'Dulcolax',
 'Morphine',
 'MoviPrep',
 'Trilafon',
 'Fluconazole',
 'Contrave',
 'Clonazepam',
 'Metaxalone',
 'Venlafaxine',
 'Ledipasvir / sofosbuvir',
 'Symbyax',
 'Tamsulosin',
 'Doxycycline',
 'Dulaglutide',
 'Intuniv',
 'Buprenorphine',
 'Qvar',
 'Opdivo',
 'Pyridium',
 'Latuda',
 'Bupropion',
 'Implanon',
 'Effexor XR',
 'Drospirenone / ethinyl estradiol',
 'NuvaRing',
 'Prepopik',
 'Tretinoin',
 'Gildess Fe 1 / 20',
 'Ethinyl estradiol / norgestimate'

In [9]:
def filter_nones(x):
    return x["condition"] is not None

In [ ]:
# None 값이 들어간 행 필터닝
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)

Filter: 100%|██████████████████████████████████████████| 53766/53766 [00:00<00:00, 100322.46 examples/s]


In [11]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}


drug_dataset.map(lowercase_condition)

Map: 100%|███████████████████████████████████████████████| 53471/53471 [00:07<00:00, 7571.69 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

In [12]:
drug_dataset["train"]["condition"][:3]

['Left Ventricular Dysfunction', 'ADHD', 'Birth Control']

In [13]:
# 프로세싱후 새로운 칼럼 만들기
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

drug_dataset = drug_dataset.map(compute_review_length)

Map: 100%|███████████████████████████████████████████████| 53471/53471 [00:07<00:00, 7088.03 examples/s]


In [14]:
# Inspect the first training example
drug_dataset["train"][0]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'Left Ventricular Dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

In [ ]:
# 단 하나의 단어만 있는 리뷰
drug_dataset["train"].sort("review_length")[:3]

{'patient_id': [111469, 13653, 53602],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse'],
 'condition': ['Hepatitis C', 'ADHD', 'Birth Control'],
 'review': ['"Headache"', '"Great"', '"Awesome"'],
 'rating': [10.0, 10.0, 10.0],
 'date': ['February 3, 2015', 'October 20, 2009', 'November 23, 2015'],
 'usefulCount': [41, 3, 0],
 'review_length': [1, 1, 1]}

In [16]:
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)

Filter: 100%|██████████████████████████████████████████| 53471/53471 [00:00<00:00, 116710.40 examples/s]

{'train': 138514, 'test': 46108}


In [ ]:
# HTML 코드 제거 필요. 웹에서 받은 데이터는 이 과정 필요할듯
import html

text = "I&#039;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

In [18]:
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

Map: 100%|███████████████████████████████████████████████| 46108/46108 [00:07<00:00, 6054.55 examples/s]


In [ ]:
# 배치 단위 전처리 가능
new_drug_dataset = drug_dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]}, batched=True
)

Map: 100%|█████████████████████████████████████████████| 46108/46108 [00:00<00:00, 208024.00 examples/s]


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True)

In [21]:
%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████████████████████████████████████████| 46108/46108 [00:04<00:00, 10107.46 examples/s]

CPU times: user 1min 46s, sys: 8.44 s, total: 1min 55s
Wall time: 18.7 s


In [ ]:
# Rust로 짜여진 Fast 버전이 없을 경우 map의 멀티프로세싱 사용
slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)


def slow_tokenize_function(examples):
    return slow_tokenizer(examples["review"], truncation=True)


tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)

Map (num_proc=8): 100%|██████████████████████████████████| 46108/46108 [00:11<00:00, 3959.36 examples/s]


In [ ]:
def tokenize_and_split(examples):
    return tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True, # truncation하고 자른 남은 부분까지 리턴
    )

In [ ]:
result = tokenize_and_split(drug_dataset["train"][0])
[len(inp) for inp in result["input_ids"]] # 두개가 나옴

[128, 49]

In [25]:
tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)

Map:   0%|                                                            | 0/138514 [00:00<?, ? examples/s]


ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

In [ ]:
# 1000개 행을 배치
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset["train"].column_names
)

Map: 100%|███████████████████████████████████████████████| 46108/46108 [00:15<00:00, 3051.46 examples/s]


In [27]:
len(tokenized_dataset["train"]), len(drug_dataset["train"])

(206772, 138514)

In [28]:
def tokenize_and_split(examples):
    result = tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result

In [29]:
tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
tokenized_dataset

Map: 100%|███████████████████████████████████████████████| 46108/46108 [00:15<00:00, 2908.19 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 206772
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 68876
    })
})

In [30]:
drug_dataset.set_format("pandas")

In [31]:
drug_dataset["train"][:3]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89


In [32]:
train_df = drug_dataset["train"][:]

In [33]:
frequencies = (
    train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "count": "frequency"})
)
frequencies.head()

,condition,frequency
0,Birth Control,27655
1,Depression,8023
2,Acne,5209
3,Anxiety,4991
4,Pain,4744


In [34]:
from datasets import Dataset

freq_dataset = Dataset.from_pandas(frequencies)
freq_dataset

Dataset({
    features: ['condition', 'frequency'],
    num_rows: 819
})

In [35]:
drug_dataset.reset_format()

In [36]:
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
# Add the "test" set to our `DatasetDict`
drug_dataset_clean["test"] = drug_dataset["test"]
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

In [37]:
drug_dataset_clean.save_to_disk("drug-reviews")

Saving the dataset (1/1 shards): 100%|█████████████████| 46108/46108 [00:00<00:00, 466994.84 examples/s]


In [38]:
from datasets import load_from_disk

drug_dataset_reloaded = load_from_disk("drug-reviews")
drug_dataset_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

In [39]:
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"drug-reviews-{split}.jsonl")

Creating json from Arrow format: 100%|█████████████████████████████████| 47/47 [00:00<00:00, 140.04ba/s]


In [40]:
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"drug-reviews-{split}.jsonl")

Creating json from Arrow format: 100%|█████████████████████████████████| 47/47 [00:00<00:00, 137.52ba/s]


In [41]:
data_files = {
    "train": "drug-reviews-train.jsonl",
    "validation": "drug-reviews-validation.jsonl",
    "test": "drug-reviews-test.jsonl",
}
drug_dataset_reloaded = load_dataset("json", data_files=data_files)

Generating train split: 110811 examples [00:00, 276464.03 examples/s]
Generating validation split: 27703 examples [00:00, 262540.26 examples/s]
Generating test split: 46108 examples [00:00, 286143.11 examples/s]


### Instruction dataset용 message 추가

In [16]:
def create_conversation(example):
    return {
        "messages": [
            {"role": "system", "content": example["system"]},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]}
        ]
    }

In [21]:
type(sub_dataset_dict)

datasets.arrow_dataset.Dataset

In [18]:
sub_dataset_dict = dataset_dict['train'].take(100)

In [ ]:
# 자동으로 칼럼으로 처리됨
processed_dataset = sub_dataset_dict.map(create_conversation)

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 3218.47 examples/s]


In [20]:
processed_dataset

Dataset({
    features: ['output', 'instruction', 'system', 'messages'],
    num_rows: 100
})

In [23]:
processed_dataset['messages'][:3]

[[{'content': 'You are a Python code analyst, evaluating scripts for potential improvements.',
   'role': 'system'},
  {'content': 'Create a nested loop to print every combination of numbers between 0-9, excluding any combination that contains the number 5. Additionally, exclude any combination that contains a repeating digit. Implement the solution without using any built-in functions or libraries to check for repeating digits.',
   'role': 'user'},
  {'content': 'Here is an example of a nested loop in Python to print every combination of numbers between 0-9, excluding any combination that contains the number 5 or repeating digits:\n\n```python\nfor i in range(10):  # First digit\n    for j in range(10):  # Second digit\n        for k in range(10):  # Third digit\n            # Checking for the conditions\n            if i != 5 and j != 5 and k != 5 and i != j and i != k and j != k:\n                print(i, j, k)\n```\n\nThis code will generate and print every combination of three di

In [30]:
# filter 함수 : true false로 필터링
def filter_function(example):
    return len(example['instruction']) > 50

filtered_dataset = processed_dataset.filter(filter_function)


Filter: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 1114.96 examples/s]


In [31]:
filtered_dataset

Dataset({
    features: ['output', 'instruction', 'system', 'messages'],
    num_rows: 100
})

In [32]:
# 열 제거
dataset = filtered_dataset.remove_columns('system')

In [33]:
dataset

Dataset({
    features: ['output', 'instruction', 'messages'],
    num_rows: 100
})

In [ ]:
# 열 추가
def add_length_column(example):
    example['length'] = len(example['text'])
    return example

dataset = dataset.map(add_length_column)

In [ ]:
# 분할
train_test = dataset.train_test_split(test_size=0.2)
# 이러면 split이 만들어짐
train_test

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'messages'],
        num_rows: 80
    })
    test: Dataset({
        features: ['output', 'instruction', 'messages'],
        num_rows: 20
    })
})

In [36]:
# 다시 split을 지정해서 또 나누면
test_valid = train_test['test'].train_test_split(test_size=0.5)

test_valid

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'messages'],
        num_rows: 10
    })
    test: Dataset({
        features: ['output', 'instruction', 'messages'],
        num_rows: 10
    })
})

In [ ]:
# 새로운 dict에 지정
datasets = {
    'train': train_test['train'],
    'validation':test_valid['train'],
    'test': test_valid['test'] 
}

datasets

{'train': Dataset({
     features: ['output', 'instruction', 'messages'],
     num_rows: 80
 }),
 'validation': Dataset({
     features: ['output', 'instruction', 'messages'],
     num_rows: 10
 }),
 'test': Dataset({
     features: ['output', 'instruction', 'messages'],
     num_rows: 10
 })}

In [ ]:
# 데이터 셔플

shuffled_dataset = dataset.shuffle(seed=777)

In [ ]:
# 배치 처리

def tokenize_function(examples):
  return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True,batch_size=1000)

In [ ]:
# 데이터셋 캐싱
# 같은 작업을 캐싱?
processed_dataset = dataset.map(preprocess_function, cache_file_name='your_dataset')

In [ ]:
# 토큰화 인코딩
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('dazare/ggobugi-llama3-v4')

def create_conversation(sample):
  return {
    "messages": tokenizer.apply_chat_template([
      {"role": "system", "content": sample["system"]},
      {"role": "user", "content": sample["instruction"]},
      {"role": "assistant", "content": sample["output"]}
      ], tokenize=False, add_generation_prompt=False)
    }


processed_dataset = dataset.map(
    create_conversation,
    remove_columns=dataset.features, # 기존 열을 제거하여 새로운 데이터 구조를 유지
    batched=False
)

In [ ]:
# 전처리 로그
def safe_preprocess(example):
    try:
        example = preprocess_function(example)
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error processing example {example['id']}: {e}\n")
    return example


dataset = dataset.map(safe_preprocess)

In [ ]:
# 로컬에서 로드할때


In [ ]:
from datasets import concatenate_datasets
combined_dataset = concatenate_datasets([dataset1, dataset2])
